In [1]:
import sys
from pathlib import Path

# Bootstrap: find the project's src/ dynamically by walking up from the current working
# directory to the first folder containing a .env file, then add its src/ to sys.path.
# Portable across machines (Windows/Mac) with no hardcoded absolute paths.
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

# Project-wide config (PROJECT_ROOT, RAW_DIR, PROCESSED_DIR, BROWSER_HEADERS, SSL_VERIFY, ...)
# and the download-log helpers, plus libraries used throughout this notebook.
from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os, re          # os: path joins vs reserved config dirs; re: parse year from kaopen_YYYY.xls link
import requests        # fetch the discovery page + the .xls file (cells 3, 5)
import pycountry       # validate / remap the file's ISO3 codes to current standard (cell 7)

# Load the download log (per-source currency/filenames) into memory.
log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 40
PROJECT_ROOT: C:\Users\mjbou\governance-framework


## 31 · Chinn-Ito KAOPEN Pipeline (capital-account openness, de jure)

**Source:** Chinn & Ito, Portland State University — KAOPEN, the first standardized principal component of the AREAER binary capital-control variables. Most-cited academic measure of de jure capital-account openness.
**Access:** **AUTOMATED.** Version + download URL discovered dynamically from the faculty page each run (newest `kaopen_YYYY.xls`) — no hardcoded year. ⚠ **FRAGILE URL** (personal faculty page `web.pdx.edu/~ito/`): if restructured, discovery fails loudly; manual fallback in `docs/instructions_data_maintenance.md`.
**Concept role:** Category 5 → *Macroeconomic policy framework quality*. One of six primaries (Romelli CBI, IMF Fiscal Rules, AREAER FARI, **KAOPEN**, iMaPP, Reinhart-Rogoff). **NOT more authoritative than AREAER FARI (nb 32)** — it is an automatable *derivative* of the same AREAER source data, used as the broad-time-series complement / cross-check.

### Fields produced
- `kaopen` (raw PCA, ≈ −1.94 to 2.28; **higher = MORE open**): the Chinn-Ito index. **PRIMARY scored field.**
- `kaopen_norm` (0–1): same index normalized (source column `ka_open`). Supplementary / convenience.

### Automated — refreshes on re-run; no manual step
Discovery scrapes the faculty page for the newest `kaopen_YYYY.xls`, parses the year from the link, downloads to `RAW_DIR`. **No MANUAL UPDATE points in the runtime.** A new release (e.g. `kaopen_2024.xls`) is picked up with no code change.

### ⚠ Version non-stability (drives full-replace)
Each release re-runs the PCA over the **entire** sample, recalculating ALL historical years; KAOPEN is **not comparable across versions**. Treatment: full-replace on every update, never append; never compare a country's KAOPEN across two framework vintages as the same number.

### Notes
- **⚠ Sign convention is OPPOSITE of FARI.** KAOPEN higher = more *open*; FARI higher = more *restrictive*. Invert one before any FARI×KAOPEN cross-check.
- **Vintage alignment:** data-year Y = AREAER report (Y+1) = end-of-Y status (2023 = AREAER 2024 = controls as of end-2023). Align FARI cross-checks on end-of-year status.
- File ships **ISO3** (`ccode`) + IMF–WB numeric (`cn`), so no fuzzy name mapping. A few legacy codes are remapped to current ISO3 (e.g. `ZAR`→`COD`); dead codes (e.g. `ANT`, Netherlands Antilles) are flagged, not invented.
- **Coverage varies by country:** latest available year < max for some (AFG ends 2000, IRQ 2002, SOM 2007, Netherlands Antilles 2009, Micronesia 2021); k1–k4 set missing for AFG/IRQ/SOM in recent years. "Latest value per country" must use each country's last non-null year.
- Direction check (from 2023 readme): open economies (HKG, SGP, USA, most OECD) ≈ +2.28 (max); closed (IRN, SYR, Sri Lanka) ≈ −1.94 (min).

### As-of (page state at authoring): 2026-06-18. MANUAL SNAPSHOT — does not auto-update; verify at run.
`kaopen_2023.xls`, coverage 1970–2023, 182 countries (readme appendix; HTML page text says 181 — readme authoritative), page "Updated January 18, 2026." Next likely update: summer/fall 2026 (`kaopen_2024.xls`, after AREAER 2025).



In [2]:
# Source identity + dynamic version discovery. SOURCE_ID keys the download log / registry.
# The faculty page lists the current data file as a year-stamped link (kaopen_YYYY.xls); we
# parse the newest year FROM the page rather than hardcoding it, so a future release
# (e.g. kaopen_2024.xls) is picked up automatically with no code change.
SOURCE_ID = "CHINN_ITO"                                       # new id; cross-referenced to IMF_AREAER role
PAGE_URL  = "https://web.pdx.edu/~ito/Chinn-Ito_website.htm"  # discovery page (⚠ fragile personal faculty page)

# Fetch the discovery page. BROWSER_HEADERS avoids naive-bot blocks; verify uses config's
# auto-detected SSL_VERIFY rather than a hardcoded True/False.
resp = requests.get(PAGE_URL, headers=BROWSER_HEADERS, timeout=30, verify=SSL_VERIFY)
resp.raise_for_status()                                       # fail loudly on 4xx/5xx

# Collect every kaopen_YYYY.xls link year and take the max. Anchoring the regex on '.xls'
# excludes the .dta file and the Readme_kaopenYYYY.pdf decoys, so only the real Excel data
# file matches; max() also picks the newest if an archived prior year is left on the page.
years = [int(y) for y in re.findall(r"kaopen_(\d{4})\.xls", resp.text)]
if not years:                                                 # page restructured / link gone
    raise RuntimeError(
        "No kaopen_YYYY.xls link found on the Chinn-Ito page. The fragile faculty URL may have "
        "changed — see the Chinn-Ito manual fallback in docs/instructions_data_maintenance.md."
    )
LATEST_YEAR  = max(years)                                     # newest published version
VERSION      = str(LATEST_YEAR)                               # version stamp = data coverage end year
DOWNLOAD_URL = f"https://web.pdx.edu/~ito/kaopen_{LATEST_YEAR}.xls"  # built from parsed year, not hardcoded
RAW_FILE     = os.path.join(RAW_DIR, f"kaopen_{LATEST_YEAR}.xls")    # local target in reserved RAW_DIR (flat)

print(f"Discovered VERSION : {VERSION}")
print(f"DOWNLOAD_URL       : {DOWNLOAD_URL}")
print(f"RAW_FILE           : {RAW_FILE}")

Discovered VERSION : 2023
DOWNLOAD_URL       : https://web.pdx.edu/~ito/kaopen_2023.xls
RAW_FILE           : C:\Users\mjbou\governance-framework\data\raw\kaopen_2023.xls


In [3]:
# Engine guard: kaopen_*.xls is legacy BIFF (.xls); pandas needs the xlrd engine to read it
# (openpyxl handles only .xlsx). Fail early with a clear install hint instead of a cryptic
# engine error inside the read cell.
try:
    import xlrd  # noqa: F401
    print(f"xlrd OK (version {xlrd.__version__})")
except ImportError:
    raise ImportError(
        "xlrd not installed in the governance-framework env. Install it (one line):  "
        "conda install -n governance-framework xlrd"
    )

# Download the discovered .xls to RAW_DIR. Streamed write; the target filename equals the
# source filename, so re-running REPLACES any existing same-name file. verify per config.
with requests.get(DOWNLOAD_URL, headers=BROWSER_HEADERS, timeout=120,
                  stream=True, verify=SSL_VERIFY) as r:
    r.raise_for_status()                                    # fail loudly on HTTP error
    with open(RAW_FILE, "wb") as f:                         # 'wb' truncates/replaces any existing file
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

print(f"Saved -> {RAW_FILE}  ({os.path.getsize(RAW_FILE):,} bytes)")

xlrd OK (version 2.0.2)
Saved -> C:\Users\mjbou\governance-framework\data\raw\kaopen_2023.xls  (1,091,584 bytes)


In [5]:
# Read Sheet1 and standardize to the framework's tidy long panel. Layout confirmed by the
# diagnostic: one sheet, already long (one row per country-year), columns
# cn | ccode | country_name | year | kaopen | ka_open.
df = pd.read_excel(RAW_FILE, engine="xlrd")

# Rename to framework field names:
#  - ccode is ISO Alpha-3 (validated in the next cell) -> country_code
#  - ka_open is the 0-1 normalization -> kaopen_norm (kills the confusing kaopen/ka_open near-dupe)
#  - country_name -> country_name_source (the source's own label, kept for traceability; matches nb 32)
# kaopen (raw PCA index) is kept verbatim. cn (IMF-WB numeric code) is dropped — not used
# downstream; the framework keys on ISO3 (matches nb 32 dropping the IFS code).
df = df.rename(columns={
    'ccode': 'country_code',
    'country_name': 'country_name_source',
    'ka_open': 'kaopen_norm',
})

# Keep keys + the two scored fields, in order; enforce dtypes; sort for a stable panel.
df = df[['country_code', 'country_name_source', 'year', 'kaopen', 'kaopen_norm']].copy()
df['year']        = df['year'].astype(int)
df['kaopen']      = pd.to_numeric(df['kaopen'], errors='coerce')
df['kaopen_norm'] = pd.to_numeric(df['kaopen_norm'], errors='coerce')
df = df.sort_values(['country_code', 'year']).reset_index(drop=True)

print(f"Standardized: {df.shape}  | countries: {df['country_code'].nunique()}  "
      f"| years: {df['year'].min()}-{df['year'].max()}")
print(f"kaopen range: {df['kaopen'].min():.3f} .. {df['kaopen'].max():.3f}  "
      f"| kaopen_norm range: {df['kaopen_norm'].min():.3f} .. {df['kaopen_norm'].max():.3f}")
print(df.head(4).to_string())

Standardized: (9891, 5)  | countries: 185  | years: 1970-2023
kaopen range: -1.939 .. 2.281  | kaopen_norm range: 0.000 .. 1.000
  country_code country_name_source  year  kaopen  kaopen_norm
0            .     Serbia, Rep. of  2006     NaN          NaN
1            .     Serbia, Rep. of  2007     NaN          NaN
2            .     Serbia, Rep. of  2008     NaN          NaN
3            .     Serbia, Rep. of  2009     NaN          NaN


In [7]:
# Resolve country codes to current ISO3, then drop empty country-years.
# Diagnostic established four non-standard codes: ZAR (DR Congo, legacy of Zaire — has data),
# ANT (Netherlands Antilles — dissolved 2010, has 1970-2009 data, NO valid successor),
# TMP (Timor-Leste — 0 data) and '.' (Serbia — 0 data). Policy:
#   - remap legacy codes that carry data to current ISO3 (ZAR->COD);
#   - keep ANT as-is (dead code, no valid successor, out of the >2M target sample anyway —
#     flagged not invented; it simply won't join downstream);
#   - drop rows with no kaopen value (matches nb 32's dropna on value), which removes the
#     empty '.' / TMP placeholders entirely along with all other empty country-years.
LEGACY_ISO3 = {
    'ZAR': 'COD',   # Zaire -> Congo, Dem. Rep. (current ISO3); carries real data, must not be lost
}
df['country_code'] = df['country_code'].replace(LEGACY_ISO3)

# Drop empty country-years (no index value). kaopen and kaopen_norm are NaN together.
before = len(df)
df = df.dropna(subset=['kaopen']).reset_index(drop=True)
print(f"Dropped {before - len(df)} empty country-year rows -> {len(df)} remain")

# Re-validate ISO3 on the cleaned panel; surface any code still non-standard.
# EXPECTED residual: ['ANT'] only (accepted dead code). Anything ELSE here is a MANUAL UPDATE:
# add a LEGACY_ISO3 mapping (if it's a real country with a current ISO3) and re-run.
def _is_valid_iso3(code):
    try:
        return pycountry.countries.get(alpha_3=str(code)) is not None
    except Exception:
        return False
valid_mask = df['country_code'].map(_is_valid_iso3)
residual = sorted(df.loc[~valid_mask, 'country_code'].unique().tolist())
print(f"Non-standard ISO3 remaining (expect ['ANT']): {residual}")
print(f"Distinct country_codes: {df['country_code'].nunique()}  "
      f"| valid ISO3 codes: {df.loc[valid_mask, 'country_code'].nunique()}")
print(f"Years: {df['year'].min()}-{df['year'].max()}  | rows: {len(df)}")

Dropped 1551 empty country-year rows -> 8340 remain
Non-standard ISO3 remaining (expect ['ANT']): ['ANT']
Distinct country_codes: 182  | valid ISO3 codes: 181
Years: 1970-2023  | rows: 8340


In [8]:
# Validation gates — fail loudly if a future export violates expectations.
# (1) Internal consistency: the version discovered from the page must equal the data's max year.
assert int(df['year'].max()) == int(VERSION), \
    f"max data year {df['year'].max()} != discovered VERSION {VERSION}"
# (2) No empty values survived the clean; both index columns fully populated.
assert df['kaopen'].notna().all() and df['kaopen_norm'].notna().all(), "unexpected NaN after clean"
# (3) Normalized index strictly within [0,1]; raw PCA within a plausible band (readme: -1.94..2.28).
assert df['kaopen_norm'].between(0, 1).all(), "kaopen_norm outside [0,1]"
assert df['kaopen'].min() > -3 and df['kaopen'].max() < 3, "kaopen outside plausible PCA range"
# (4) Every country_code is a 3-char code (valid ISO3, or the one accepted dead code ANT).
assert df['country_code'].str.len().eq(3).all(), "non-3-char country_code present"
print("All validation gates passed.")

# Direction spot-check (NOT an assertion — sign convention is easy to flip vs FARI):
# higher kaopen = MORE open. Latest available year for a few known-open vs known-closed economies.
spot = (df[df['country_code'].isin(['HKG', 'SGP', 'USA', 'CHN', 'IRN', 'SYR'])]
        .sort_values('year').groupby('country_code').tail(1)
        [['country_code', 'year', 'kaopen', 'kaopen_norm']]
        .sort_values('kaopen', ascending=False))
print(spot.to_string(index=False))

All validation gates passed.
country_code  year    kaopen  kaopen_norm
         SGP  2023  2.280830     1.000000
         USA  2023  2.280830     1.000000
         HKG  2023  2.280830     1.000000
         CHN  2023 -1.253876     0.162432
         IRN  2023 -1.939372     0.000000
         SYR  2023 -1.939372     0.000000


In [9]:
# Assemble the final tidy panel (already standardized + validated) and write the processed
# output. Full-overwrite each run — required by KAOPEN's version non-stability (never append).
final = df[['country_code', 'country_name_source', 'year', 'kaopen', 'kaopen_norm']].copy()
output_path = os.path.join(PROCESSED_DIR, "chinn_ito_clean.csv")
final.to_csv(output_path, index=False)
print(f"Written: {output_path}  | shape: {final.shape}")

# Currency DERIVED from the data (latest year present), never hardcoded. KAOPEN data-year Y
# corresponds to AREAER report (Y+1) and reflects end-of-Y capital-control status.
latest_year    = int(final['year'].max())
areaer_report  = latest_year + 1
retrieval_date = datetime.today().strftime("%Y-%m-%d")

# Record the automated download in the log.
update_entry(
    SOURCE_ID,
    last_successful_download_date=retrieval_date,
    data_as_of_date=f"{latest_year} (AREAER {areaer_report}; end-{latest_year} status)",
    local_filename="chinn_ito_clean.csv",
    latest_available_version=f"KAOPEN {latest_year} (kaopen_{latest_year}.xls)",
    notes=("Chinn-Ito KAOPEN capital-account openness (de jure). PRIMARY tier-1 (Macro policy "
           "framework quality). AUTOMATED: faculty page scraped for newest kaopen_YYYY.xls, year "
           "parsed dynamically (no hardcode); fragile personal-page URL. kaopen (raw PCA, "
           "higher=MORE open; OPPOSITE sign to AREAER FARI) primary; kaopen_norm (0-1) supplementary. "
           f"181 valid-ISO3 + ANT (dead code) = 182; 1970-{latest_year}. ZAR->COD remap; Serbia/Timor "
           "unscored, dropped. VERSION NON-STABLE: PCA recomputed each release -> full-replace, never "
           "append. Derivative / cross-check to AREAER FARI (nb 32), not more authoritative.")
)
print_entry(SOURCE_ID)

Written: C:\Users\mjbou\governance-framework\data\processed\chinn_ito_clean.csv  | shape: (8340, 5)
[download_log] Updated entry for CHINN_ITO
  source_id: CHINN_ITO
  last_attempted_date: 2026-06-18
  last_successful_download_date: 2026-06-18
  data_as_of_date: 2023 (AREAER 2024; end-2023 status)
  local_filename: chinn_ito_clean.csv
  latest_available_version: KAOPEN 2023 (kaopen_2023.xls)
  no_update_reason: nan
  notes: Chinn-Ito KAOPEN capital-account openness (de jure). PRIMARY tier-1 (Macro policy framework quality). AUTOMATED: faculty page scraped for newest kaopen_YYYY.xls, year parsed dynamically (no hardcode); fragile personal-page URL. kaopen (raw PCA, higher=MORE open; OPPOSITE sign to AREAER FARI) primary; kaopen_norm (0-1) supplementary. 181 valid-ISO3 + ANT (dead code) = 182; 1970-2023. ZAR->COD remap; Serbia/Timor unscored, dropped. VERSION NON-STABLE: PCA recomputed each release -> full-replace, never append. Derivative / cross-check to AREAER FARI (nb 32), not more a